# EcoHome Energy Advisor - RAG Setup

In this notebook, you'll set up the Retrieval-Augmented Generation (RAG) pipeline for the EcoHome Energy Advisor. This will allow the agent to access and cite relevant energy-saving tips and best practices.

## Learning Objectives
- Set up ChromaDB vector store
- Load and process energy-saving documents
- Create embeddings for document chunks
- Implement semantic search functionality
- Test the RAG pipeline

## Documents
All `.txt` files in `data/documents/` are loaded automatically. Add new documents there and re-run this notebook to include them in the vector store.


## 1. Import Required Libraries


In [1]:
# Import the necessary libraries for RAG setup
import os
from langchain_chroma  import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

## 2. Load and Process Documents


In [3]:
# Load all .txt documents from the data/documents/ directory automatically
documents = []
doc_dir = "data/documents"

txt_files = sorted(f for f in os.listdir(doc_dir) if f.endswith(".txt"))

for filename in txt_files:
    doc_path = os.path.join(doc_dir, filename)
    loader = TextLoader(doc_path)
    docs = loader.load()
    documents.extend(docs)
    print(f"Loaded {len(docs)} document(s) from {doc_path}")

print(f"\nTotal documents loaded: {len(documents)}")


Loaded 1 document(s) from data/documents/tip_device_best_practices.txt
Loaded 1 document(s) from data/documents/tip_energy_savings.txt
Loaded 1 document(s) from data/documents/tip_energy_storage.txt
Loaded 1 document(s) from data/documents/tip_hvac_optimization.txt
Loaded 1 document(s) from data/documents/tip_renewable_integration.txt
Loaded 1 document(s) from data/documents/tip_seasonal_energy.txt
Loaded 1 document(s) from data/documents/tip_smart_home_automation.txt

Total documents loaded: 7


## 3. Split Documents into Chunks


In [4]:
# Split documents into smaller chunks for better retrieval
# Use RecursiveCharacterTextSplitter with appropriate chunk_size and chunk_overlap
# Experiment with different chunk sizes (e.g., 500, 1000, 1500 characters)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Split the documents
splits = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(splits)} chunks")

# Show sample chunk
if splits:
    print(f"\nSample chunk (first 200 characters):")
    print(splits[0].page_content[:200] + "...")


Split 7 documents into 30 chunks

Sample chunk (first 200 characters):
Large devices like electric vehicles, washing machines and dishwashers often support delayed start or timer functions. Schedule these devices to run outside of peak electricity pricing hours or during...


## 4. Create Vector Store


In [5]:
import shutil
import chromadb

persist_directory = "data/vectorstore"

# Fully remove any previous vectorstore so ChromaDB always starts with a clean slate.
# (Chroma 0.6+ can fail with "readonly database" if stale WAL/journal files exist.)
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)
os.makedirs(persist_directory)

# Initialize embeddings
embeddings = OpenAIEmbeddings(
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("OPENAI_API_KEY"),  # type: ignore
)

# Use a PersistentClient so the SQLite file is created fresh with correct permissions
chroma_client = chromadb.PersistentClient(path=persist_directory)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    client=chroma_client,
)

print(f"Vector store created and persisted to {persist_directory}")
print(f"Total vectors stored: {len(splits)}")


Vector store created and persisted to data/vectorstore
Total vectors stored: 30


## 5. Test the RAG Pipeline


In [6]:
# Test the search functionality
# Try different queries related to energy optimization
# Test queries like:
# - "electric vehicle charging tips"
# - "thermostat optimization"
# - "dishwasher energy saving"
# - "solar power maximization"

test_queries = [
    "electric vehicle charging tips",
    "thermostat optimization",
    "dishwasher energy saving",
    "solar power maximization",
    "HVAC system efficiency",
    "pool pump scheduling"
]

print("=== Testing Vector Search ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    docs = vectorstore.similarity_search(query, k=2)
    for i, doc in enumerate(docs):
        print(f"  Result {i+1}: {doc.page_content[:100]}...")


=== Testing Vector Search ===

Query: 'electric vehicle charging tips'
  Result 1: Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
  Result 2: Optimal Charging Strategies:
- Solar charging: configure the battery to charge from solar surplus fi...

Query: 'thermostat optimization'
  Result 1: HVAC systems typically account for 40–50% of a home's total energy bill, making them the single best...
  Result 2: Thermostat Setpoints by Season:
- Summer cooling target: 78°F (26°C) when home, 85°F (29°C) when awa...

Query: 'dishwasher energy saving'
  Result 1: Dishwasher Best Practices:
- Only run when completely full
- Use the energy-saving or eco mode when ...
  Result 2: Automating Appliances Around Electricity Prices:
- Connect smart plugs to dishwashers, washing machi...

Query: 'solar power maximization'
  Result 1: Maximising Self-Consumption:
- A 7–10 kWh home battery stores midday surplus for use in the evening ...
  Result 2: M

## 6. Test the Search Tool


In [7]:
# Test the search_energy_tips tool from tools.py
# Import and test the tool with various queries
# Verify that it returns relevant results

from tools import search_energy_tips

# Test the search_energy_tips function
print("=== Testing search_energy_tips Tool ===")

test_queries = [
    "electric vehicle charging",
    "thermostat settings",
    "dishwasher optimization",
    "solar power tips"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    result = search_energy_tips.invoke(
        input={
            "query": query, 
            "max_results": 3,
        }
    )
    
    if "error" in result:
        print(f"  Error: {result['error']}")
    else:
        print(f"  Found {result['total_results']} results")
        for i, tip in enumerate(result['tips']):
            print(f"    {i+1}. {tip['content'][:100]}...")
            print(f"       Source: {tip['source']}")
            print(f"       Relevance: {tip['relevance_score']}")


=== Testing search_energy_tips Tool ===

Query: 'electric vehicle charging'
  Found 3 results
    1. Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
       Source: data/documents/tip_device_best_practices.txt
       Relevance: high
    2. Optimal Charging Strategies:
- Solar charging: configure the battery to charge from solar surplus fi...
       Source: data/documents/tip_energy_storage.txt
       Relevance: high
    3. Integration with Solar and Smart Home:
- A solar-plus-storage system should prioritise: (1) direct s...
       Source: data/documents/tip_energy_storage.txt
       Relevance: medium

Query: 'thermostat settings'
  Found 3 results
    1. Thermostat Setpoints by Season:
- Summer cooling target: 78°F (26°C) when home, 85°F (29°C) when awa...
       Source: data/documents/tip_hvac_optimization.txt
       Relevance: high
    2. Winter (December–February):
- Heating accounts for over 60% of energy bills in colder clima